In [9]:
"""
Entity resolution, STEP 2: validate the (reviewed/edited) candidate
groups from step1_generate_entity_candidates.py.

REWRITTEN from the partition-based version. That version let the LLM
invent its own sub-group boundaries within a group_id, which produced
a real problem: it correctly split some members out (e.g. Amide A,
Amide B) but then incorrectly re-merged others it had just as much
evidence against (Amide I with Amide II), silently restructuring a
group you had deliberately assembled for review. The safety flag
caught it before it reached the trusted output, but you shouldn't have
to re-discover something already established.

New contract, three rules:
  1. The LLM evaluates each group_id AS YOU DEFINED IT. It never
     splits it into sub-groups or invents a different arrangement.
     One row in, one row out, always.
  2. It either agrees the whole group is one real entity, or it
     disagrees. Disagreement doesn't discard the group, it flags it
     for your review with a specific reason.
  3. A canonical name is ALWAYS proposed, even on disagreement, so you
     always see what the LLM would suggest, rather than a blank field.

Only agreed AND unflagged groups make it into approved_lookup_table.
Disagreed or flagged groups stay visible in candidate_groups with their
canonical name and justification, for you to decide.

Requires OPENAI_API_KEY as an environment variable.
Designed for Jupyter/Colab execution. No __main__ guard.
"""

import os
import re
import json
import time
import pandas as pd
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv(override=True)

# ---------------------------------------------------------------
# CONFIG
# ---------------------------------------------------------------
REVIEWED_CANDIDATES_PATH = "entity_candidate_groups_for_review.xlsx"  # edit this file by hand first if needed
LLM_MODEL = "gpt-4o-mini"

OUTPUT_XLSX = "entity_resolution_review.xlsx"

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

# ---------------------------------------------------------------
# GENERAL asymmetric-word check (unchanged mechanism from before):
# words present in some group members but not all, no fixed word list,
# generalizes to any future distinguishing word.
# ---------------------------------------------------------------
_STOPWORDS = {"from", "of", "the", "a", "an", "and", "or", "in", "on", "at", "to", "for", "with"}


def tokenize(entity_name):
    words = re.findall(r"[a-z0-9\-]+", str(entity_name).lower())
    return {w for w in words if w not in _STOPWORDS}


def get_asymmetric_tokens(members):
    token_sets = [tokenize(m) for m in members]
    shared_core = set.intersection(*token_sets) if token_sets else set()
    all_tokens = set.union(*token_sets) if token_sets else set()
    return sorted(all_tokens - shared_core)


def has_order_only_difference(members):
    """Same members, different word order, e.g. 'X followed by Y' vs
    'Y followed by X'. Zero asymmetric tokens would otherwise hide this."""
    token_sets = [frozenset(tokenize(m)) for m in members]
    raw_texts = [str(m).strip().lower() for m in members]
    for i in range(len(members)):
        for j in range(i + 1, len(members)):
            if token_sets[i] == token_sets[j] and raw_texts[i] != raw_texts[j]:
                return True
    return False


# ---------------------------------------------------------------
# LOAD REVIEWED CANDIDATE GROUPS
# ---------------------------------------------------------------
candidates_df = pd.read_excel(REVIEWED_CANDIDATES_PATH)
print(f"Loaded {len(candidates_df)} entities across {candidates_df['group_id'].nunique()} candidate groups")

group_sizes = candidates_df.groupby("group_id").size()
valid_group_ids = group_sizes[group_sizes >= 2].index
skipped = group_sizes[group_sizes < 2].index
if len(skipped) > 0:
    print(f"Skipping {len(skipped)} groups reduced to a single member during review")

# ---------------------------------------------------------------
# LLM: EVALUATE EACH GROUP AS A WHOLE, NEVER RESTRUCTURE IT
# ---------------------------------------------------------------
VALIDATION_PROMPT = """You are validating a candidate synonym group for entity resolution in a
plant protein functional properties knowledge graph. This group was assembled by embedding
similarity, then reviewed and deliberately arranged by the researcher exactly as shown below.

Candidate group (do NOT reorder, split, or regroup these, evaluate them exactly as given):
{members}
{asymmetric_note}
Task: decide whether EVERY entity in this group is genuinely the SAME real-world entity
referred to differently (true synonyms, abbreviations, or spelling/formatting variants).

- If you agree all members are the same entity: set agrees_with_grouping to true, and
  propose ONE canonical name (prefer the most complete, standard scientific form).
- If you do NOT agree, because at least one member is a genuinely distinct entity (a
  different specific method, material, sample, or named measurement, e.g. Amide I is a
  different spectral band from Amide II, even if a subset of the group is otherwise
  the same thing): set agrees_with_grouping to false. You must still propose a
  canonical_name, your best-guess name for what the majority/core entities represent, so
  the researcher sees your reasoning even in disagreement. In justification, name
  SPECIFICALLY which entity or entities you believe do not belong and why.

  IMPORTANT for canonical_name on disagreement: represent everything genuinely shared by
  ALL members, even if the SPECIFIC WORDING of one shared component varies. Do not drop a
  component just because its exact form differs, use a generic term for it instead. For
  example, if every member combines "alkaline hydrogen peroxide extraction" with some form
  of ultrasound (ultrasonication / ultrasound / ultrasound-assisted), the canonical name
  must still reflect the combined treatment, e.g. "alkaline hydrogen peroxide extraction
  with ultrasound treatment", NOT just "alkaline hydrogen peroxide extraction" alone.
  Dropping a universally-present component produces a canonical name that misrepresents
  what the group actually contains, even under disagreement about the specific variant.

Do not invent a different grouping arrangement. You are evaluating the group as a whole,
not partitioning it.

If a word-difference list is shown above, address whether each one reflects a real
distinction or is inconsequential phrasing, this should directly inform your agree/disagree
decision.

Respond ONLY with JSON, no markdown fences:
{{"agrees_with_grouping": true/false, "canonical_name": "...", "justification": "..."}}
"""


def validate_group(members):
    asymmetric_tokens = get_asymmetric_tokens(members)
    if asymmetric_tokens:
        asymmetric_note = (
            "\nWord-difference check: these words appear in SOME but not ALL of the "
            f"entities above: {', '.join(asymmetric_tokens)}. Address whether each one "
            "signals a real distinction, and let that inform your agree/disagree decision.\n"
        )
    else:
        asymmetric_note = ""

    prompt = VALIDATION_PROMPT.format(
        members="\n".join(f"- {m}" for m in members),
        asymmetric_note=asymmetric_note,
    )
    resp = client.chat.completions.create(
        model=LLM_MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
    )
    raw = resp.choices[0].message.content.strip()
    time.sleep(0.2)
    try:
        result = json.loads(raw)
    except json.JSONDecodeError:
        return {"agrees_with_grouping": False, "canonical_name": None,
                 "justification": f"PARSE_FAILED: {raw[:200]}"}

    # canonical_name is mandatory per the contract, even on disagreement,
    # a missing one on agreement is treated as a parse problem, not applied
    if result.get("agrees_with_grouping") and not result.get("canonical_name"):
        result["justification"] = ("MISSING_CANONICAL_NAME despite agreement: "
                                     + str(result.get("justification", "")))
        result["agrees_with_grouping"] = False
    return result


results = []
for gid in valid_group_ids:
    members = candidates_df.loc[candidates_df["group_id"] == gid, "entity"].tolist()
    verdict = validate_group(members)

    agrees = bool(verdict.get("agrees_with_grouping"))
    canonical_name = verdict.get("canonical_name")
    justification = verdict.get("justification")

    asymmetric_tokens = get_asymmetric_tokens(members)
    order_only = has_order_only_difference(members)
    # flagged for review whenever the LLM disagrees, OR it agrees but an
    # unresolved word-difference or order-only difference remains, same
    # general (non-word-specific) safety net as before
    needs_review = bool((not agrees) or asymmetric_tokens or order_only)

    results.append({
        "group_id": gid,
        "members": "; ".join(members),
        "asymmetric_tokens": "; ".join(asymmetric_tokens),
        "is_synonym_group": agrees,
        "proposed_canonical_name": canonical_name,  # always populated, per the contract
        "justification": justification,
        "needs_manual_review": needs_review,
    })

review_df = pd.DataFrame(results)
n_agreed = review_df["is_synonym_group"].sum()
n_disagreed = (~review_df["is_synonym_group"]).sum()
n_flagged = review_df["needs_manual_review"].sum()
print(f"\nLLM agreed with {n_agreed} groups, disagreed with {n_disagreed}, "
      f"{n_flagged} flagged for manual review total")

# ---------------------------------------------------------------
# BUILD FLAT LOOKUP TABLE, ONLY high-confidence merges
# (is_synonym_group = True AND needs_manual_review = False)
# ---------------------------------------------------------------
lookup_rows = []
confident = review_df[review_df["is_synonym_group"] & (~review_df["needs_manual_review"])]
for _, row in confident.iterrows():
    canonical = row["proposed_canonical_name"]
    for m in row["members"].split("; "):
        lookup_rows.append({"raw_name": m, "canonical_name": canonical})

lookup_df = pd.DataFrame(lookup_rows)

# ---------------------------------------------------------------
# SAVE
# ---------------------------------------------------------------
readme_rows = [
    "HOW TO READ THIS FILE",
    "",
    f"This file validates candidate groups from {REVIEWED_CANDIDATES_PATH}, EXACTLY as you",
    "arranged them. The LLM never splits, reorders, or regroups a group_id, it only",
    "evaluates the group you gave it as a whole and either agrees or disagrees.",
    "",
    "- group_id: your group, unchanged, one row here per group_id, no exceptions.",
    "",
    "- members: the entity names in this group, exactly as you arranged them.",
    "",
    "- asymmetric_tokens: words present in SOME but not ALL members, computed automatically,",
    "  no fixed word list. Shown to the LLM before it decides.",
    "",
    "- is_synonym_group: True only if the LLM agreed EVERY member is the same entity. False",
    "  means the LLM believes at least one member does not belong, check justification for",
    "  which one and why, your group arrangement itself is never altered by this disagreement.",
    "",
    "- proposed_canonical_name: ALWAYS populated, agreement or not, so you always see what",
    "  the LLM would call this group even when it disagrees with including everyone in it.",
    "",
    "- justification: names specifically which entity the LLM objects to, when it disagrees.",
    "",
    "- needs_manual_review: True whenever the LLM disagreed, OR it agreed but an unresolved",
    "  word-difference or order-only difference remains. Not included in",
    "  approved_lookup_table below regardless of which reason triggered it.",
    "",
    "- approved_lookup_table sheet: raw_name -> canonical_name, ready to apply, only for",
    "  groups that were BOTH agreed AND not flagged. Spot-check before trusting.",
]
readme_df = pd.DataFrame({"": readme_rows})

with pd.ExcelWriter(OUTPUT_XLSX) as writer:
    readme_df.to_excel(writer, sheet_name="READ_ME_FIRST", index=False)
    review_df.to_excel(writer, sheet_name="candidate_groups", index=False)
    lookup_df.to_excel(writer, sheet_name="approved_lookup_table", index=False)

print(f"\nSaved to {OUTPUT_XLSX}")
print("Spot-check approved_lookup_table before applying it to your main triples file.")

Loaded 727 entities across 385 candidate groups
Skipping 168 groups reduced to a single member during review

LLM agreed with 57 groups, disagreed with 160, 213 flagged for manual review total

Saved to entity_resolution_review.xlsx
Spot-check approved_lookup_table before applying it to your main triples file.
